<a href="https://colab.research.google.com/github/sasindu345/OctWave3/blob/main/notebooks/02_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Evidence & Decisions

Run this **after** every training run, before touching `logs/EXPERIMENTS.md`.

Its job is to answer one question honestly: *did that change actually help, or did
a number move by chance?* The rules it enforces are in [AGENTS.md](../AGENTS.md).

No CPU/GPU needed — this reads saved out-of-fold predictions, so it runs fine on a
CPU runtime while a training job holds the GPU elsewhere.

## Setup

In [3]:
import os, sys
from pathlib import Path

REPO = '/content/OctWave3'
if 'google.colab' in sys.modules:
    if not os.path.exists(REPO):
        !git clone -q https://github.com/sasindu345/OctWave3.git $REPO
    else:
        !cd $REPO && git pull -q
    sys.path.insert(0, REPO)
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    OUT = Path('/content/drive/MyDrive/octwave3/outputs')
else:
    sys.path.insert(0, str(Path.cwd().parent))
    OUT = Path('../outputs')

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.config import cfg
from src.analysis import *
from src.utils import load_oof

cfg.out_dir = OUT
use_house_style()
print('OOF files:', sorted(p.name for p in (OUT / 'oof').glob('*.npz')))

fatal: could not read Username for 'https://github.com': No such device or address
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ModuleNotFoundError: No module named 'src'

## 1. Single experiment — the four-panel report

Learning curves (overfitting), fold spread, confusion matrix (where errors are),
and reliability (are the probabilities honest). Title carries the 95% CI, because
a point estimate on its own invites over-reading.

In [ ]:
EXP    = 'exp01_baseline'
FOLDS  = [0]                                   # extend to [0,1,2,3,4] when 5 folds trained
classes = ['0 (neither)', '1 (Tom)', '2 (Jerry)', '3 (both)']

run_log = pd.read_json(OUT / 'run_log.jsonl', lines=True)
fig = evidence_report(cfg, EXP, FOLDS, classes, run_log,
                      out_png=OUT / f'figures/{EXP}.png')


## 2. How uncertain is that score?

If a rival model's score falls inside this interval, you have no evidence they differ.

In [ ]:
probs, targets = load_oof(OUT, EXP, FOLDS[0])
for m in ['macro_f1', 'accuracy']:
    ci = bootstrap_ci(targets, probs, metric=m)
    print(f"{m:>9}: {ci['point']:.4f}  95% CI [{ci['lo']:.4f}, {ci['hi']:.4f}]  (width {ci['width']:.4f})")


## 3. Is the candidate actually better?

**The gate.** Paired bootstrap + McNemar on the same validation samples.
Only an `ADOPT` verdict makes the candidate the new baseline.

In [ ]:
BASE = 'exp01_baseline'
CAND = 'exp02_5fold_b0'
FOLD = 0

probs_a, targets = load_oof(OUT, BASE, FOLD)
probs_b, _       = load_oof(OUT, CAND, FOLD)

result = decide(targets, probs_a, probs_b, BASE, CAND, metric='macro_f1')


## 4. Fold spread — the noise band made visible

Heavily overlapping boxes mean the difference in means is not real.

In [ ]:
FOLDS = [0, 1, 2, 3, 4]
# Only summarize experiments that have OOF files available
available_exps = [e for e in ['exp01_baseline', 'exp02_5fold_b0']
                  if (OUT / 'oof' / f'{e}_f0.npz').exists()]
summaries = [cv_summary(OUT, e, [f for f in FOLDS if (OUT / 'oof' / f'{e}_f{f}.npz').exists()], metric='macro_f1')
             for e in available_exps]
if summaries:
    plot_fold_box(summaries)
    plt.show()
    for s in summaries:
        print(f"{s['exp']:>22}: {s['mean']:.4f} ± {s['std']:.4f}   folds={np.round(s['scores'], 4)}")
else:
    print("No OOF files found yet for summary.")


## 5. Where are the errors?

Drives the *next* experiment: a hot off-diagonal cell is a targeted fix, not 'train longer'.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
plot_confusion(targets, probs, classes, ax=ax[0])
plot_per_class_f1(targets, probs, classes, ax=ax[1])
plt.tight_layout(); plt.show()


## 6. Can I trust my CV?

**The most important chart in the competition.** Fill in `logs/EXPERIMENTS.md` as
you submit, then plot CV against public LB.

- `r > 0.7` → CV is trustworthy; iterate offline and stop burning submissions.
- low / negative `r` → the validation split is wrong. Fix that before tuning anything.

In [ ]:
exps = pd.DataFrame([
    # {'exp': 'exp01', 'cv': 0.0000, 'lb': 0.0000},
])
if len(exps):
    plot_cv_vs_lb(exps); plt.show()
else:
    print('no submissions logged yet - fill this in from logs/EXPERIMENTS.md')

## 7. Record it

Paste the `decide()` output into `logs/DECISIONS.md` using the template there, add
a row to `logs/EXPERIMENTS.md`, and commit. An unlogged experiment is a wasted one —
you will otherwise re-run it in three days having forgotten the result.

## 8. E1: OOF Error Analysis (Diagnostic)\nRun these cells to identify the dominant error patterns before making any model changes.

In [ ]:
import numpy as np
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)
from src.utils import load_oof
from src.config import cfg

all_probs, all_targets = [], []
for f in range(5):
    p, t = load_oof(cfg.out_dir, "exp02_5fold_b0", f)
    all_probs.append(p)
    all_targets.append(t)

probs = np.concatenate(all_probs)       # (2680, 4)
targets = np.concatenate(all_targets)   # (2680,)
preds = probs.argmax(1)

print(f"OOF Macro F1:  {f1_score(targets, preds, average='macro'):.4f}")
print(f"OOF Accuracy:  {accuracy_score(targets, preds):.4f}")
print(f"OOF Weighted F1: {f1_score(targets, preds, average='weighted'):.4f}")
print()
names = ['0:neither', '1:Tom', '2:Jerry', '3:both']
print(classification_report(targets, preds, target_names=names, digits=4))

In [ ]:
cm = confusion_matrix(targets, preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

print("=== Raw Confusion Matrix ===")
print(cm)
print("\n=== Row-Normalized Confusion Matrix ===")
print(np.round(cm_norm, 3))

print("\n=== Error Pairs (True → Pred, count, % of true class) ===")
errors = []
for i in range(4):
    for j in range(4):
        if i != j and cm[i,j] > 0:
            errors.append((cm[i,j], i, j, cm_norm[i,j]))
for count, i, j, pct in sorted(errors, reverse=True):
    print(f"  True {names[i]} → Pred {names[j]}: {count:4d} ({pct*100:5.1f}%)")

In [ ]:
import matplotlib.pyplot as plt

max_conf = probs.max(axis=1)
correct = (preds == targets)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(max_conf[correct], bins=30, alpha=0.7, label='Correct', color='#0072B2')
axes[0].hist(max_conf[~correct], bins=30, alpha=0.7, label='Wrong', color='#D55E00')
axes[0].set_xlabel('Max Predicted Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence: Correct vs Incorrect')
axes[0].legend()

for c in range(4):
    mask = (targets == c) & (~correct)
    if mask.sum() > 0:
        axes[1].hist(max_conf[mask], bins=20, alpha=0.5, label=f'True={names[c]}' )
axes[1].set_xlabel('Max Predicted Probability')
axes[1].set_title('Confidence When Wrong (by true class)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
wrong_mask = preds != targets
wrong_conf = max_conf[wrong_mask]
wrong_idx = np.where(wrong_mask)[0]
worst = wrong_idx[np.argsort(-wrong_conf)[:20]]

print("=== Top 20 Highest-Confidence Errors ===")
for idx in worst:
    print(f"  img={idx:5d} true={names[targets[idx]]:12s} "
          f"pred={names[preds[idx]]:12s} conf={max_conf[idx]:.3f} "
          f"probs={np.round(probs[idx], 3)}")

In [ ]:
per_class_f1 = f1_score(targets, preds, average=None)
per_class_errors = np.zeros(4)
per_class_total = np.zeros(4)
for c in range(4):
    mask = targets == c
    per_class_total[c] = mask.sum()
    per_class_errors[c] = (preds[mask] != c).sum()

print("=== Per-Class Analysis ===")
print(f"{'Class':<15} {'N':>5} {'Errors':>7} {'Error%':>7} {'F1':>7} {'F1 Gap':>7}")
macro_f1 = per_class_f1.mean()
for c in range(4):
    gap = per_class_f1[c] - macro_f1
    print(f"{names[c]:<15} {int(per_class_total[c]):>5} "
          f"{int(per_class_errors[c]):>7} {per_class_errors[c]/per_class_total[c]*100:>6.1f}% "
          f"{per_class_f1[c]:>6.4f} {gap:>+6.4f}")
print(f"\nMacro F1 = {macro_f1:.4f}")
print(f"\nThe class dragging Macro F1 down most: "
      f"{names[np.argmin(per_class_f1)]} (F1={per_class_f1.min():.4f})")

In [ ]:
import cv2
from src.dataset import build_dataframe

df = build_dataframe(cfg)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
worst_20 = wrong_idx[np.argsort(-wrong_conf)[:20]]

for ax, idx in zip(axes.flat, worst_20):
    img = cv2.imread(str(df.path.iloc[idx]))
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
    ax.set_title(f"T:{names[targets[idx]]} P:{names[preds[idx]]}\n"
                 f"conf={max_conf[idx]:.2f}", fontsize=8)
    ax.axis('off')

plt.suptitle('Top 20 Most Confident Errors (exp02_5fold_b0)', fontsize=14)
plt.tight_layout()
plt.show()